# Online Retail: Data Cleaning, EDA and Slow-Moving Inventory Analysis

**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter
**Data:** UCI Online Retail (about 540K transactions, Dec 2010 to Dec 2011, UK-based online gift retailer)

**Flow:** load -> clean (with an audit log) -> EDA -> slow-moving inventory classification -> findings.

**Before running:** download the dataset and place it in `data/` as `Online Retail.xlsx` (UCI) or `OnlineRetail.csv` (Kaggle copy).
Every number in the final summary is computed by the notebook, so quote *those* numbers in your resume and README.

In [1]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 50

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ---- analysis parameters (tune here) ----
DORMANT_DAYS = 90       # no sale in more than this many days (before the dataset ends) => dormant
OUTLIER_QTY = 50_000    # single-line quantity treated as an order-and-cancel outlier

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 1. Load data

In [2]:
xlsx = DATA_DIR / "Online Retail.xlsx"
csv = DATA_DIR / "OnlineRetail.csv"
dtypes = {"InvoiceNo": str, "StockCode": str}

if xlsx.exists():
    raw = pd.read_excel(xlsx, dtype=dtypes)          # needs openpyxl
elif csv.exists():
    raw = pd.read_csv(csv, dtype=dtypes, encoding="ISO-8859-1")
else:
    raise FileNotFoundError("Put 'Online Retail.xlsx' or 'OnlineRetail.csv' inside the data/ folder.")

print(f"Rows: {len(raw):,}   Columns: {raw.shape[1]}")
raw.head()

Rows: 541,909   Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [3]:
raw.info()
print("\nMissing values per column:")
print(raw.isna().sum())
raw.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(1), str(3)
memory usage: 33.1+ MB

Missing values per column:


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"541,909.00",NaN,NaN,NaN,9.55,"-80,995.00",1.00,3.00,10.00,"80,995.00",218.08
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,"541,909.00",NaN,NaN,NaN,4.61,"-11,062.06",1.25,2.08,4.13,"38,970.00",96.76
CustomerID,"406,829.00",NaN,NaN,NaN,"15,287.69","12,346.00","13,953.00","15,152.00","16,791.00","18,287.00","1,713.60"
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Cleaning
Each step is logged so the cleaning is auditable. Order: de-duplicate -> separate cancellations -> drop non-product codes -> drop invalid rows -> drop order-and-cancel outliers.

*Missing `CustomerID` rows are kept on purpose: this analysis is product-level, not customer-level.*

In [4]:
log = {"raw_rows": len(raw)}

# 2.1 exact duplicate rows
df = raw.drop_duplicates().copy()
log["exact_duplicates_removed"] = log["raw_rows"] - len(df)

# 2.2 tidy types / text
df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip().str.upper()
df["Description"] = df["Description"].str.strip()
df["Country"] = df["Country"].str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(f"Duplicates removed: {log['exact_duplicates_removed']:,}")

Duplicates removed: 5,268


In [5]:
# 2.3 separate cancellations (invoice numbers starting with 'C')
is_cancel = df["InvoiceNo"].str.startswith("C")
cancellations = df[is_cancel].copy()
sales = df[~is_cancel].copy()
log["cancellation_rows_separated"] = len(cancellations)
print(f"Cancellation rows: {len(cancellations):,}   Sales rows: {len(sales):,}")

Cancellation rows: 9,251   Sales rows: 527,390


In [6]:
# 2.4 non-product stock codes (postage, fees, manual adjustments, samples, gift vouchers...)
# Real products have codes starting with 5 digits (e.g. 85123A, 22423). Anything else is a non-product code.
def is_non_product(codes):
    return ~codes.str.match(r"^\d{5}")

np_mask = is_non_product(sales["StockCode"])
print("Non-product rows in sales, by code:")
print(sales.loc[np_mask, "StockCode"].value_counts().head(15))

log["non_product_rows_removed"] = int(np_mask.sum())
sales = sales[~np_mask].copy()
cancellations = cancellations[~is_non_product(cancellations["StockCode"])].copy()

Non-product rows in sales, by code:
StockCode
POST            1130
DOT              709
M                323
C2               142
DCGSSGIRL         13
BANK CHARGES      12
DCGSSBOY          11
GIFT_0001_20      10
GIFT_0001_10       9
GIFT_0001_30       8
DCGS0003           5
GIFT_0001_50       4
PADS               4
GIFT_0001_40       3
B                  3
Name: count, dtype: int64


In [7]:
# 2.5 invalid sales rows (zero/negative price or quantity: write-offs, adjustments)
bad = (sales["Quantity"] <= 0) | (sales["UnitPrice"] <= 0)
log["invalid_qty_or_price_removed"] = int(bad.sum())
sales = sales[~bad].copy()
print(f"Invalid rows removed: {int(bad.sum()):,}")

Invalid rows removed: 2,480


In [8]:
# 2.6 order-and-cancel outliers: a huge order that is fully reversed distorts top-product revenue
big_sales = sales[sales["Quantity"] >= OUTLIER_QTY]
big_cancels = cancellations[cancellations["Quantity"].abs() >= OUTLIER_QTY]
print("Outlier sales lines:")
display(big_sales[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]])
print("Matching cancellation lines:")
display(big_cancels[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "InvoiceDate"]])

log["outlier_sales_rows_removed"] = len(big_sales)
log["outlier_cancel_rows_removed"] = len(big_cancels)
sales = sales.drop(big_sales.index)
cancellations = cancellations.drop(big_cancels.index)

Outlier sales lines:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,2011-01-18 10:01:00
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,2011-12-09 09:15:00


Matching cancellation lines:


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,InvoiceDate
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,2011-01-18 10:17:00
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2.08,2011-12-09 09:27:00


In [9]:
# 2.7 feature engineering
sales["Revenue"] = sales["Quantity"] * sales["UnitPrice"]
sales["Month"] = sales["InvoiceDate"].dt.to_period("M")
sales["Weekday"] = sales["InvoiceDate"].dt.day_name()
sales["Hour"] = sales["InvoiceDate"].dt.hour
cancellations["Value"] = cancellations["Quantity"] * cancellations["UnitPrice"]

log["clean_sales_rows"] = len(sales)
log_df = pd.Series(log, name="rows").to_frame()
display(log_df)

print(f"\nClean sales rows : {len(sales):,}")
print(f"Unique products  : {sales['StockCode'].nunique():,}")
print(f"Unique invoices  : {sales['InvoiceNo'].nunique():,}")
print(f"Date range       : {sales['InvoiceDate'].min():%Y-%m-%d} -> {sales['InvoiceDate'].max():%Y-%m-%d}")
print(f"Total revenue    : {sales['Revenue'].sum():,.0f}")

,rows
raw_rows,541909
exact_duplicates_removed,5268
cancellation_rows_separated,9251
non_product_rows_removed,2406
invalid_qty_or_price_removed,2480
outlier_sales_rows_removed,2
outlier_cancel_rows_removed,2
clean_sales_rows,522502



Clean sales rows : 522,502
Unique products  : 3,790


Unique invoices  : 19,771
Date range       : 2010-12-01 -> 2011-12-09
Total revenue    : 10,001,168


## 3. Cancellations

In [10]:
cancel_value = cancellations["Value"].abs().sum()
print(f"Cancellation lines : {len(cancellations):,}")
print(f"Cancelled value    : {cancel_value:,.0f}  ({cancel_value / sales['Revenue'].sum():.1%} of sales revenue)")

top_cancel = (cancellations.groupby(["StockCode", "Description"])["Quantity"].sum().abs()
              .sort_values(ascending=False).head(10))
top_cancel

Cancellation lines : 8,666
Cancelled value    : 230,248  (2.3% of sales revenue)


StockCode  Description                        
84347      ROTATING SILVER ANGELS T-LIGHT HLDR    9376
21108      FAIRY CAKE FLANNEL ASSORTED COLOUR     3150
85123A     WHITE HANGING HEART T-LIGHT HOLDER     2578
21175      GIN + TONIC DIET METAL SIGN            2030
22920      HERB MARKER BASIL                      1527
22273      FELTCRAFT DOLL MOLLY                   1447
47566B     TEA TIME PARTY BUNTING                 1424
15034      PAPER POCKET TRAVELING FAN             1385
20971      PINK BLUE FELT CRAFT TRINKET BOX       1321
84077      WORLD WAR 2 GLIDERS ASSTD DESIGNS      1200
Name: Quantity, dtype: int64

## 4. Exploratory data analysis

### 4.1 Monthly revenue and seasonality

In [11]:
monthly = sales.groupby("Month")["Revenue"].sum()
total_rev = monthly.sum()

sepnov_rev = sales.loc[sales["InvoiceDate"].dt.month.isin([9, 10, 11]), "Revenue"].sum()
sepnov_share = sepnov_rev / total_rev
y2011 = sales[(sales["InvoiceDate"] >= "2011-01-01") & (sales["InvoiceDate"] < "2011-12-01")]
sepnov_share_2011 = sepnov_rev / y2011["Revenue"].sum()

print(f"Sep-Nov share of ALL revenue in dataset      : {sepnov_share:.1%}")
print(f"Sep-Nov share of Jan-Nov 2011 revenue        : {sepnov_share_2011:.1%}")

colors = ["#d95f02" if p.month in (9, 10, 11) else "#7f9fbf" for p in monthly.index]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(monthly.index.astype(str), monthly.values, color=colors)
ax.set_title("Monthly revenue (Sep-Nov highlighted; Dec 2011 is a partial month)")
ax.set_ylabel("Revenue")
plt.xticks(rotation=45)
savefig("01_monthly_revenue")

Sep-Nov share of ALL revenue in dataset      : 35.8%
Sep-Nov share of Jan-Nov 2011 revenue        : 40.8%


### 4.2 Country mix

In [12]:
country_rev = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
uk_share = country_rev["United Kingdom"] / country_rev.sum()
print(f"United Kingdom share of revenue: {uk_share:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
country_rev.head(10).sort_values().plot.barh(ax=axes[0], color="#4c78a8")
axes[0].set_title("Top 10 countries by revenue")
country_rev.drop("United Kingdom").head(10).sort_values().plot.barh(ax=axes[1], color="#59a14f")
axes[1].set_title("Top 10 excluding the UK")
for a in axes:
    a.set_xlabel("Revenue")
savefig("02_country_mix")

United Kingdom share of revenue: 84.8%


### 4.3 Top products by revenue (after outlier removal)

In [13]:
desc = (sales.groupby("StockCode")["Description"]
        .agg(lambda s: s.dropna().mode().iloc[0] if s.notna().any() else ""))

top_products = (sales.groupby("StockCode")["Revenue"].sum()
                .sort_values(ascending=False).head(10))
labels = [f"{c} | {desc[c][:28]}" for c in top_products.index]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels[::-1], top_products.values[::-1], color="#e15759")
ax.set_title("Top 10 products by revenue")
ax.set_xlabel("Revenue")
savefig("03_top_products")

### 4.4 Weekday x hour demand

In [14]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (sales.groupby(["Weekday", "Hour"])["InvoiceNo"].nunique()
        .unstack(fill_value=0))
heat = heat.reindex([d for d in order if d in heat.index])

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(heat, cmap="YlOrRd", linewidths=0.3, cbar_kws={"label": "Unique invoices"}, ax=ax)
ax.set_title("Demand by weekday and hour (unique invoices)")
savefig("04_weekday_hour_heatmap")

print("Weekdays with no sales:", [d for d in order if d not in heat.index])
print("Busiest weekday:", heat.sum(axis=1).idxmax(), "| Busiest hour:", heat.sum(axis=0).idxmax())

Weekdays with no sales: ['Saturday']
Busiest weekday: Thursday | Busiest hour: 12


## 5. Slow-moving inventory analysis
Each product (StockCode) is classified by **days since its last sale**, measured from the last date in the data.

| Status | Days since last sale |
|---|---|
| Active | 0-30 |
| Slow | 31-60 |
| Very slow | 61 to `DORMANT_DAYS` (90) |
| Dormant | more than `DORMANT_DAYS` (90) |

*Caveat: the data covers about 13 months and has no stock levels. "Dormant" means "no sales in the window", so seasonal items and discontinued lines need a human check before any clearance decision.*

In [15]:
snapshot = sales["InvoiceDate"].max().normalize()

prod = (sales.groupby("StockCode")
        .agg(first_sale=("InvoiceDate", "min"),
             last_sale=("InvoiceDate", "max"),
             orders=("InvoiceNo", "nunique"),
             units=("Quantity", "sum"),
             revenue=("Revenue", "sum"))
        .join(desc.rename("Description")))

prod["days_since_last_sale"] = (snapshot - prod["last_sale"].dt.normalize()).dt.days

labels_status = ["Active", "Slow", "Very slow", "Dormant"]
prod["status"] = pd.cut(prod["days_since_last_sale"],
                        bins=[-1, 30, 60, DORMANT_DAYS, np.inf],
                        labels=labels_status)

summary = (prod.groupby("status", observed=True)
           .agg(products=("revenue", "size"), revenue=("revenue", "sum")))
summary["pct_products"] = summary["products"] / summary["products"].sum() * 100
summary["pct_revenue"] = summary["revenue"] / summary["revenue"].sum() * 100
summary = summary.reindex(labels_status)
display(summary)

n_products = len(prod)
dormant = prod[prod["status"] == "Dormant"]
dormant_pct_products = len(dormant) / n_products
dormant_pct_revenue = dormant["revenue"].sum() / prod["revenue"].sum()
print(f"Products classified : {n_products:,}")
print(f"Dormant products    : {len(dormant):,} ({dormant_pct_products:.1%} of products, "
      f"{dormant_pct_revenue:.1%} of revenue)")

,products,revenue,pct_products,pct_revenue
status,,,,
Active,2884,"9,526,678.67",76.09,95.26
Slow,185,"161,422.10",4.88,1.61
Very slow,86,"120,603.54",2.27,1.21
Dormant,635,"192,463.36",16.75,1.92


Products classified : 3,790
Dormant products    : 635 (16.8% of products, 1.9% of revenue)


In [16]:
# Sensitivity: how the dormant count changes with the threshold
rows = []
for d in [60, 90, 120, 180, 270]:
    m = prod["days_since_last_sale"] > d
    rows.append({"threshold_days": d, "dormant_products": int(m.sum()),
                 "pct_products": m.mean() * 100,
                 "pct_revenue": prod.loc[m, "revenue"].sum() / prod["revenue"].sum() * 100})
pd.DataFrame(rows)

,threshold_days,dormant_products,pct_products,pct_revenue
0,60,721,19.02,3.13
1,90,635,16.75,1.92
2,120,568,14.99,1.48
3,180,427,11.27,1.01
4,270,172,4.54,0.25


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

x = np.arange(len(labels_status))
w = 0.38
axes[0].bar(x - w / 2, summary["pct_products"], w, label="% of products", color="#4c78a8")
axes[0].bar(x + w / 2, summary["pct_revenue"], w, label="% of revenue", color="#f58518")
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_status)
axes[0].set_ylabel("Percent")
axes[0].set_title("Product share vs revenue share by status")
axes[0].legend()

axes[1].hist(prod["days_since_last_sale"], bins=40, color="#7f9fbf")
axes[1].axvline(DORMANT_DAYS, color="red", ls="--", label=f"Dormant > {DORMANT_DAYS} days")
axes[1].set_title("Days since last sale (per product)")
axes[1].set_xlabel("Days")
axes[1].legend()
savefig("05_slow_moving_status")

In [18]:
# Clearance review list: dormant products, longest-idle first
clearance = (dormant.assign(avg_unit_price=lambda d: d["revenue"] / d["units"])
             .sort_values("days_since_last_sale", ascending=False)
             .reset_index()
             [["StockCode", "Description", "first_sale", "last_sale", "days_since_last_sale",
               "orders", "units", "revenue", "avg_unit_price"]])
clearance.to_csv(OUT_DIR / "clearance_candidates.csv", index=False)
prod.reset_index().to_csv(OUT_DIR / "product_status_all.csv", index=False)
print(f"Saved {len(clearance):,} clearance candidates -> {OUT_DIR / 'clearance_candidates.csv'}")
clearance.head(15)

Saved 635 clearance candidates -> outputs/clearance_candidates.csv


,StockCode,Description,first_sale,last_sale,days_since_last_sale,orders,units,revenue,avg_unit_price
0,82615,PINK MARSHMALLOW SCARF KNITTING KIT,2010-12-01 16:11:00,2010-12-01 16:11:00,373,1,36,91.80,2.55
1,84854,GIRLY PINK TOOL SET,2010-12-01 09:41:00,2010-12-01 09:41:00,373,1,5,24.75,4.95
2,21268,VINTAGE BLUE TINSEL REEL,2010-12-02 15:46:00,2010-12-02 15:46:00,372,1,2,0.84,0.42
3,35271S,GOLD PRINT PAPER BAG,2010-12-01 17:35:00,2010-12-02 13:49:00,372,2,19,3.61,0.19
4,22275,WEEKEND BAG VINTAGE ROSE PAISLEY,2010-12-02 16:51:00,2010-12-02 16:51:00,372,1,69,527.85,7.65
5,16161M,WRAP PINK FLOCK,2010-12-02 18:15:00,2010-12-02 18:15:00,372,1,25,10.50,0.42
6,72803B,OCEAN SCENT CANDLE JEWELLED DRAWER,2010-12-03 11:28:00,2010-12-03 11:28:00,371,1,1,8.47,8.47
7,84614A,PINK BAROQUE FLOCK CANDLE HOLDER,2010-12-03 15:45:00,2010-12-03 15:45:00,371,1,2,11.90,5.95
8,47579,TEA TIME BREAKFAST BASKET,2010-12-05 13:00:00,2010-12-05 13:00:00,369,1,1,2.10,2.10
9,84963A,PINK PAINTED KASHMIRI CHAIR,2010-12-05 11:57:00,2010-12-05 11:57:00,369,1,4,171.80,42.95


## 6. Key findings (auto-generated from the results above)

In [19]:
top_prod_code = top_products.index[0]
print(f"""
DATA CLEANING
- Started with {log['raw_rows']:,} transactions; removed {log['exact_duplicates_removed']:,} exact duplicates.
- Separated {log['cancellation_rows_separated']:,} cancellation lines; removed {log['non_product_rows_removed']:,} non-product rows
  and {log['invalid_qty_or_price_removed']:,} invalid (zero/negative) rows.
- Excluded {log['outlier_sales_rows_removed']} order-and-cancel outlier sale(s) and {log['outlier_cancel_rows_removed']} matching cancellation(s).
- Clean sales table: {log['clean_sales_rows']:,} rows.

SLOW-MOVING INVENTORY
- Classified {n_products:,} products by days since last sale.
- {len(dormant):,} dormant products (> {DORMANT_DAYS} days): {dormant_pct_products:.0%} of products, {dormant_pct_revenue:.0%} of revenue.

DEMAND PATTERNS
- Sep-Nov = {sepnov_share:.0%} of all revenue ({sepnov_share_2011:.0%} of Jan-Nov 2011).
- UK = {uk_share:.0%} of revenue.
- Top product by revenue: {top_prod_code} ({desc[top_prod_code]}).
""")


DATA CLEANING
- Started with 541,909 transactions; removed 5,268 exact duplicates.
- Separated 9,251 cancellation lines; removed 2,406 non-product rows
  and 2,480 invalid (zero/negative) rows.
- Excluded 2 order-and-cancel outlier sale(s) and 2 matching cancellation(s).
- Clean sales table: 522,502 rows.

SLOW-MOVING INVENTORY
- Classified 3,790 products by days since last sale.
- 635 dormant products (> 90 days): 17% of products, 2% of revenue.

DEMAND PATTERNS
- Sep-Nov = 36% of all revenue (41% of Jan-Nov 2011).
- UK = 85% of revenue.
- Top product by revenue: 22423 (REGENCY CAKESTAND 3 TIER).

